# 第 18 课：RTF 实时率、延迟、吞吐与基准测试

本课目标：不再用一个 RTF 数字概括全部性能，能够设计可信的流式 ASR benchmark。

<!-- course-upgrade-v2 -->
## 学习导航

| 项目 | 内容 |
|---|---|
| 所属阶段 | 流式 ASR |
| 建议投入 | 3～5 小时，可分 2～3 次完成 |
| 前置要求 | 完成第 17 课；如果前测低于 2/3，先回看上一课小结 |
| 本课核心 | RTF、first/final latency、P50/P99 |
| 完成标准 | 能口头解释核心概念；独立完成强化题；从空白重写核心函数 |

高效顺序：**先回答前测 → 预测代码结果 → 再运行 → 修改一个变量 → 关闭答案复现 → 次日回忆。**


<!-- course-upgrade-v2 -->
## 课前诊断（先不要运行代码）

1. 分别用一句话解释：RTF、first/final latency、P50/P99。
2. 画出这三个概念之间的输入—输出关系。
3. 写下你最不确定的一点，并给出一个暂时猜测。

自评：答对 0～1 题先复习前置课；答对 2 题可以正常学习；3 题都能讲清楚则直接挑战代码和迁移题。


<!-- course-bridge-v3 -->
## 知识接力：先取回旧知识，再进入本课

### 3 分钟闭卷回忆

在新 Markdown cell 中回答，**不要先翻前文**：CTC collapse 与 prefix 状态；因果/非因果上下文；整段结果的基线。

- 三项都能用“含义 + 单位/shape + 一个数字例子”回答：进入本课。
- 能回答两项：学习本课，但把缺口记入 `LEARNING_LOG.md`。
- 只能回答零到一项：先回到 [上一课](17_PGS动态修正_apd_rpl_rg.ipynb)与[唯一学习路径](../LEARNING_PATH.md)，做一次最小实验；不要靠继续看新术语掩盖断点。

### 本课接口契约

```text
输入：按时间到达的 chunk、前端/模型/解码器状态
  ↓ 本课要学会的变换、状态或判断
输出：可与离线对照的 partial/final、状态更新和延迟指标
```

学完后必须能解释：输入的哪个单位/shape/状态若丢失，会让输出“仍能运行却语义错误”。


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

def find_root():
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "pyproject.toml").exists(): return p
    raise FileNotFoundError("请从 learn_asr 或 notebooks 目录启动 Jupyter")

ROOT = find_root()
BLANK = "∅"
plt.rcParams["figure.figsize"] = (11, 4)
print("项目根目录:", ROOT)

import time
from ipywidgets import interact, FloatSlider

## 1. RTF 定义

$$RTF=\frac{\text{处理时间}}{\text{音频时长}}$$

- RTF=1：处理 1 秒音频需要 1 秒；
- RTF=0.2：处理 10 秒音频约需 2 秒；
- RTF>1：单路处理追不上音频输入。

注意：离线批处理 RTF、单路流式 RTF、服务器并发吞吐不是同一个指标。

In [ ]:
@interact(audio_s=FloatSlider(min=1,max=60,value=10,step=1),rtf=FloatSlider(min=.05,max=2,value=.3,step=.05))
def rtf_calc(audio_s=10,rtf=.3):
    compute=audio_s*rtf
    print(f"音频 {audio_s:.1f}s × RTF {rtf:.2f} = 计算 {compute:.2f}s")
    print("单路速度判断:","快于实时" if rtf<1 else "无法追上实时" if rtf>1 else "刚好实时")

## 2. 实测代码必须 warm-up 和重复

In [ ]:
rng=np.random.default_rng(0); A=rng.normal(size=(256,256)).astype(np.float32);B=A.copy()
for _ in range(3): A@B
times=[]
for _ in range(20):
    t0=time.perf_counter(); A@B; times.append(time.perf_counter()-t0)
print("median ms",np.median(times)*1000,"p90 ms",np.percentile(times,90)*1000)

## 3. RTF 低，不代表用户立刻看到文字

端到端延迟可能包括：采集 chunk、右上下文、排队、特征、模型、解码、网络、稳定策略和 endpoint。

In [ ]:
rng=np.random.default_rng(4)
components={"chunk wait":160,"right context":80,"network":rng.normal(35,12,1000),"compute":rng.normal(45,10,1000),"stabilize":rng.gamma(2,45,1000)}
total=components["chunk wait"]+components["right context"]+components["network"]+components["compute"]+components["stabilize"]
plt.hist(total,bins=35);plt.axvline(np.percentile(total,50),color="C1",label="P50");plt.axvline(np.percentile(total,99),color="C3",label="P99")
plt.xlabel("End-to-end latency (ms)");plt.ylabel("Requests");plt.title("Simulated latency distribution");plt.legend();plt.show()
for p in [50,90,95,99]: print(f"P{p}={np.percentile(total,p):.1f} ms")

## 4. 推荐记录的指标

- 单路流式 RTF、并发总吞吐；
- first partial、first stable、final latency；
- P50/P90/P99，不只平均值；
- CER/WER 与延迟联合曲线；
- CPU/GPU、线程数、batch、beam、音频长度、是否含 I/O；
- 冷启动与热启动分别测试。

## 本课测试

1. 60 秒音频处理 12 秒，RTF 是多少？
2. RTF=0.1 是否保证首字延迟小于 100 ms？
3. 为什么要报告 P99？
4. batch=32 的离线 RTF 能否代表单路流式？
5. beam 变大通常会怎样影响准确率、RTF 和延迟？

<details><summary>展开参考答案</summary>

1. 0.2。2. 不能，系统可能等待 chunk、未来上下文或 endpoint。3. 平均值会隐藏尾部慢请求。4. 不能。5. 搜索更充分可能提高准确率，但通常增加计算、RTF 和延迟。

</details>

<!-- course-upgrade-v2 -->
## 强化练习：第 18 课专属题库

请先把答案写进新的 Markdown/Code cell，再展开自评标准。

### A. 基础回忆

1. 不看上文，分别定义 `RTF`、`first/final latency`、`P50/P99`。
2. 哪一个量/状态是本课最容易在模块边界丢失的？它的单位和 shape 是什么？
3. 本课至少写出两个“看起来能运行，但结果其实错误”的例子。

### B. 预测与推理

4. 场景：**平均 RTF 很低但排队很长**。先预测现象，再说明原因，最后给出一项可以验证猜测的指标。
5. 改变本课最关键参数的 0.5×、1×、2×，分别预测准确率、延迟、内存或数值误差怎样变化。
6. 画一张最小数据流图，在每条边标出 dtype、shape、时间单位或概率/代价方向。

### C. 编程与排错

7. 编程任务：**写 warm-up、重复、分位数 benchmark**。至少加入正常、边界、错误输入三类测试。
8. 故意制造一个 off-by-one、shape、状态未 reset 或数值稳定性错误；记录错误现象和定位过程。
9. 不看本课实现，从空白 cell 重写最核心函数，并用原实现作数值对照。

### D. 迁移与表达

10. 跨课任务：**把 chunk、右上下文和 endpoint 纳入延迟预算**。
11. 用 90 秒向没有学过 ASR 的人解释本课；禁止只念术语，必须举一个数字或生活例子。
12. 写出一个生产系统中会监控的指标，以及它异常时优先检查的三处位置。

<details><summary>展开自评标准</summary>

- 每题 0～2 分：0=无法回答；1=方向正确但缺少单位、边界或验证；2=解释完整且能用代码/数字验证。
- 24 分满分：达到 19 分再进入下一课；15～18 分次日重做错题；低于 15 分回看本课图和核心代码。
- 第 4 题必须包含“预测—原因—指标”，第 7～9 题必须真正运行测试，第 10 题必须明确上下游 contract。
- 核心答案至少应正确使用：RTF、first/final latency、P50/P99。

</details>


<!-- course-upgrade-v2 -->
## 间隔复习与离场票

### 离场票（现在完成）

- [ ] 我能不用笔记解释 RTF、first/final latency、P50/P99。
- [ ] 我能说出本课最常见的错误及其观测现象。
- [ ] 我能从空白重写一个核心函数，并通过至少 3 个测试。
- [ ] 我能说明本课对上一层和下一层接口的影响。

### 复习时间表

- **明天（5 分钟）**：闭卷写出三个核心概念和一个公式/shape。
- **7 天后（15 分钟）**：重做第 4、7、10 题，不运行原答案。
- **30 天后（20 分钟）**：从真实音频或随机张量重新构造一个最小实验。

把错题记录到根目录 `LEARNING_LOG.md`。不要只写“不会”，要写：原判断、证据、正确规则、下次检查动作。
